In [39]:
import pandas as pd
import numpy as np

In this notebook, we will prepare our raw merge 2 file for regression performance. We start by importing the file as is.

In [40]:
merge3 = pd.read_csv("csv_data/MERGE3.csv")
merge3.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel', 'execid', 'year', 'salary',
       'bonus', 'tdc1', 'stock_awards_fv', 'cash_comp', 'pct_equity',
       'opt_unex_exer_est_val', 'opt_unex_unexer_est_val'],
      dtype='object')

In [41]:
merge3.isna().sum()

costat                       0
curcd                        0
datafmt                      0
indfmt                       0
consol                       0
sic                          0
datadate                     0
gvkey                        0
conm                         0
tic                          0
fyear                        0
at                           0
ceq                          0
dltt                         0
lse                          0
ni                           0
revt                         0
xrd                         87
csho                         0
prcc_f                       0
sich                         3
mkt_cap                      0
industry                     0
boardid                      0
companyid                    0
datestartrole                0
directorid                   0
directorname                 0
companyname                  0
rolename                     0
dateendrole                  0
datestartroleflag            0
dateendr

In [42]:
merge3['xrd'] = merge3['xrd'].fillna(0)
merge3_clean = merge3.dropna(subset=['execid', 'year', 'salary', 'bonus', 'tdc1', 'stock_awards_fv', 'cash_comp', 'pct_equity', 'opt_unex_exer_est_val', 'opt_unex_unexer_est_val'])
print(merge3_clean.shape)

(635, 57)


We need to get some additional data from compustat. We get our tickers for this pull below:

We gather retained earnings from Compustat and bring them in:

In [43]:
retained_earnings = pd.read_csv("csv_data/retained_earnings.csv")

In [44]:
retained_earnings['fyear'] = pd.to_datetime(retained_earnings['datadate']).dt.year

merge3_clean = merge3_clean.merge(
    retained_earnings[['gvkey', 'fyear', 're']],
    on=['gvkey', 'fyear'],
    how='left'
)

print(merge3_clean.shape)
print(merge3_clean['re'].isna().sum())

(635, 58)
0


Now we are going to compute volatility:

In [45]:
volatility = pd.read_csv("csv_data/volatility.csv")
volatility['date'] = pd.to_datetime(volatility['DlyCalDt'])
volatility['fyear'] = volatility['date'].dt.year

vol_annual = volatility.groupby(['Ticker', 'fyear'])['DlyRet'].std().reset_index()
vol_annual.columns = ['tic', 'fyear', 'volatility']

merge3_clean = merge3_clean.merge(
    vol_annual[['tic', 'fyear', 'volatility']],
    on=['tic', 'fyear'],
    how='left'
)

print(merge3_clean.shape)
print(merge3_clean['volatility'].isna().sum())

(635, 59)
17


In [46]:
comp_reg = merge3_clean.copy()

In [47]:
comp_reg.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel', 'execid', 'year', 'salary',
       'bonus', 'tdc1', 'stock_awards_fv', 'cash_comp', 'pct_equity',
       'opt_unex_exer_est_val', 'opt_unex_unexer_est_val', 're', 'volatility'],
      dtype='object')

In [48]:
# DEPENDENT VARIABLE
comp_reg["roa"] = comp_reg["ni"] / comp_reg["at"]
comp_reg['adjusted_roa'] = comp_reg['roa'] - comp_reg.groupby('fyear')['roa'].transform('mean')

# KING ET AL relevant controlls
comp_reg['firm_size'] = np.log(comp_reg['at'])
comp_reg["equity_capital"] = comp_reg["ceq"] / comp_reg["at"]
charter_ratio = comp_reg['mkt_cap'] / comp_reg['ceq']
comp_reg['charter_value'] = np.where(charter_ratio > 0, np.log(charter_ratio), np.nan)# volatility
comp_reg["retained_earnings"] = comp_reg["re"] / comp_reg["at"]
comp_reg["volatility"] = comp_reg["volatility"]
# macro conditions

# my added controls
# R&D
comp_reg['xrd'] = comp_reg['xrd'].fillna(0)
comp_reg['rd_intensity'] = comp_reg['xrd'] / comp_reg['at']
#age and gender
comp_reg["ceo_age"] = comp_reg["fyear"] - pd.to_datetime(comp_reg['dob']).dt.year
comp_reg['CEO_gender'] = comp_reg['gender'].map({'M': 0, 'F': 1})

# compensation controls:
comp_reg["log_tdc1"] = np.log(comp_reg["tdc1"])
comp_reg["pct_cash_comp"] = comp_reg["cash_comp"] / comp_reg["tdc1"]
comp_reg['delta_proxy'] = (comp_reg['opt_unex_exer_est_val'] + comp_reg['opt_unex_unexer_est_val']) / comp_reg['tdc1']
comp_reg['vega_proxy'] = comp_reg['stock_awards_fv'] / comp_reg['tdc1']

print(comp_reg['charter_value'].isna().sum())


12


/Users/thefleok/Desktop/GitHub/QMSS_CEO_Thesis/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [49]:
comp_reg.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel', 'execid', 'year', 'salary',
       'bonus', 'tdc1', 'stock_awards_fv', 'cash_comp', 'pct_equity',
       'opt_unex_exer_est_val', 'opt_unex_unexer_est_val', 're', 'volatility',
       'roa', 'adjusted_roa', 'firm_size', 'equity_capital', 'charter_value',
       'retained_earnings', 'rd_intensity', 'ceo_age', 'CEO_gender',
       'log_tdc1', 'pct_cash_comp', 'delta_proxy', 'vega_proxy'],
      dtype='object')

In [51]:
# Keep only what we need
keep_cols = [
    # Identifiers
    'gvkey', 'fyear', 'industry', 'tic',
    # Dependent variable
    'adjusted_roa',
    # Raw education for factor analysis
    'UG', 'top20_ug', 'MBA', 'top20_mba', 
    'PhD', 'top20_phd', 'MD', 'top20_md', 
    "Master's", 'top20_masters',
    # Compensation
    'pct_equity', 'delta_proxy', 'vega_proxy', 'log_tdc1', 'pct_cash_comp',
    # Firm controls
    'firm_size', 'equity_capital', 'charter_value', 
    'retained_earnings', 'rd_intensity', 'volatility',
    # CEO controls
    'ceo_age', 'CEO_gender'
]

comp_reg_clean = comp_reg[keep_cols].dropna(subset=['CEO_gender', 'charter_value'])
print(comp_reg_clean.shape)

# Factor analysis
edu_vars = ['UG', 'top20_ug', 'MBA', 'top20_mba', 
            'PhD', 'top20_phd', 'MD', 'top20_md', 
            "Master's", 'top20_masters']

comp_reg_clean[edu_vars] = comp_reg_clean[edu_vars].fillna(0)

scaler = StandardScaler()
edu_scaled = scaler.fit_transform(comp_reg_clean[edu_vars])

fa = FactorAnalysis(n_components=5, random_state=42, rotation='varimax')
factors = fa.fit_transform(edu_scaled)

loadings = pd.DataFrame(
    fa.components_.T,
    index=edu_vars,
    columns=['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5']
)
print(loadings.round(3))

(617, 28)
               Factor1  Factor2  Factor3  Factor4  Factor5
UG               0.175   -0.316    0.090   -0.215    0.501
top20_ug         0.239    0.046    0.212   -0.002    0.298
MBA             -0.127   -0.044    0.694   -0.163    0.050
top20_mba       -0.045   -0.054    0.975   -0.016    0.011
PhD              0.078    0.096   -0.076    0.900   -0.090
top20_phd        0.366    0.090   -0.026    0.603    0.319
MD              -0.001    0.971   -0.061    0.073   -0.035
top20_md         0.160    0.458   -0.020    0.195    0.352
Master's         0.585   -0.090   -0.102    0.219   -0.214
top20_masters    0.968    0.025   -0.070    0.089    0.097


Factor 1 is heavily connected to the master's degree (0.955 and 0559). Factor 2 is heavily connected to the MD (0.958 and 0.372). Factor 3 is heavily connected to the top20 phd (0.944 and 0.549 for phd). Factor 4 is heavily connected to the MBA (0.751 and 0.884). Factor 5 is heavily connected to undergrad (0.416 and 0.476).

In [53]:
comp_reg_clean['masters_factor'] = factors[:, 0]
comp_reg_clean['md_factor'] = factors[:, 1]
comp_reg_clean['mba_factor'] = factors[:, 2]
comp_reg_clean['phd_factor'] = factors[:, 3]
comp_reg_clean['ug_factor'] = factors[:, 4]

print(comp_reg_clean[['ug_factor', 'mba_factor', 'phd_factor', 
                     'md_factor', 'masters_factor']].describe())

          ug_factor    mba_factor    phd_factor     md_factor  masters_factor
count  6.170000e+02  6.170000e+02  6.170000e+02  6.170000e+02    6.170000e+02
mean  -2.879022e-17 -6.909654e-17  4.030631e-17 -3.454827e-17   -7.773361e-17
std    7.227595e-01  9.784715e-01  9.241556e-01  9.761306e-01    9.765112e-01
min   -1.675277e+00 -6.279205e-01 -9.241562e-01 -4.084916e-01   -7.684326e-01
25%   -3.760624e-01 -6.203375e-01 -4.122392e-01 -1.989941e-01   -4.640735e-01
50%    1.188893e-01 -5.048238e-01 -3.997712e-01 -1.953867e-01   -4.059594e-01
75%    3.927986e-01 -2.265585e-01 -2.890732e-01 -1.196192e-01   -3.075064e-01
max    3.895132e+00  1.855958e+00  2.772605e+00  5.065214e+00    2.526857e+00


In [54]:
comp_reg_clean.isna().sum()

gvkey                 0
fyear                 0
industry              0
tic                   0
adjusted_roa          0
UG                    0
top20_ug              0
MBA                   0
top20_mba             0
PhD                   0
top20_phd             0
MD                    0
top20_md              0
Master's              0
top20_masters         0
pct_equity            0
delta_proxy           0
vega_proxy            0
log_tdc1              0
pct_cash_comp         0
firm_size             0
equity_capital        0
charter_value         0
retained_earnings     0
rd_intensity          0
volatility           16
ceo_age               0
CEO_gender            0
masters_factor        0
md_factor             0
mba_factor            0
phd_factor            0
ug_factor             0
dtype: int64

In [55]:
BASELINE1 = comp_reg_clean.to_csv("csv_data/COMP1.csv", index=False)